# Modeling

## 1. Setup
Import libraries for data handling (pandas, numpy), modeling (scikit-learn), and load the cleaned dataset from the EDA stage.

In [19]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("clean_listings.csv")

## 2. Data Preparation
We reapply the cleaning decisions made during EDA: dropping size_m2 (over 90% missing), dropping rows with missing property_type (only 3 rows), grouping sparse sub-cities into "Other", and capping bedrooms at 8 to exclude parsing outliers (e.g. a 20-bedroom listing).

In [20]:
# size_m2 was >90% missing across all sub-cities in EDA — not usable as a feature
df = df.drop(columns="size_m2")

# Only 3 rows missing property_type — safe to drop rather than impute
df = df.dropna(subset="property_type")

In [21]:
# Group sparse sub-cities (<60 listings) into "Other" to avoid overfitting on small samples
common_subcities = ["Bole", "Kirkos", "Yeka", "Nifas Silk-Lafto"]

def location_group(subcity):
    if subcity in common_subcities:
        return subcity
    else :
        return "Other"
df["location_group"]=df["sub_city"].apply(location_group)

In [22]:
# Cap bedrooms at 8 — values above this were identified as parsing outliers in EDA
df = df[df["bedrooms"] <= 8]

df["location_group"].value_counts()
df.head()
df.shape

(787, 7)

## 3. Encoding
We select our four final features and one-hot encode the two categorical ones (location_group, property_type). The target variable price_etb is kept separate, and we also create a log-transformed version to test later.

In [24]:
# Select final features and target
features = df[["location_group", "property_type", "bedrooms", "furnished"]]
target = df["price_etb"]

# drop="first" avoids the dummy variable trap — if a listing isn't Kirkos, Yeka,
# Nifas Silk-Lafto, or Other, it must be Bole, so no separate column is needed for Bole
encoder = OneHotEncoder(sparse_output=False, drop="first")
encoded = encoder.fit_transform(features[["location_group", "property_type"]])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(["location_group", "property_type"]))

# Reset indexes so concat aligns rows correctly (encoded_df has a fresh 0,1,2... index)
X = pd.concat([features[["bedrooms", "furnished"]].reset_index(drop=True), encoded_df], axis=1)

# Ensure furnished is numeric (0/1), not boolean, for consistency across all columns
X["furnished"] = X["furnished"].astype(int)

print(X.shape)
display(X.head())

(787, 10)


,bedrooms,furnished,location_group_Kirkos,location_group_Nifas Silk-Lafto,location_group_Other,location_group_Yeka,property_type_Duplex,property_type_House,property_type_Penthouse,property_type_Villa
0,3,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,3,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,3,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
# Reset target index to match X's reset index, and create a log-transformed
# version of price to test whether it improves model performance later
Y = target.reset_index(drop=True)
y_log = np.log(Y)

## 4. Train/Test Split
We split both the raw price and log price targets into training and test sets using the same random_state for a fair comparison, with 20% of data held out for testing.

In [27]:
# Two parallel splits — one for raw price, one for log price — so we can
# directly compare which target the model predicts more accurately
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)


print("X_train shape: ", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape: ", y_train.shape)
print("Y_test shape: ", y_test.shape)




X_train shape:  (629, 10)
X_test shape:  (158, 10)
y_train shape:  (629,)
Y_test shape:  (158,)


## 5. Baseline Linear Regression
We fit a baseline Linear Regression model on raw price, then repeat the process on log-transformed price to compare which target gives better predictions.

In [7]:
linreg_model=LinearRegression()
linreg_model.fit(X_train,y_train)
prediction=linreg_model.predict(X_test)

rmse=np.sqrt(mean_squared_error(y_test,prediction))
r2=r2_score(y_test,prediction)

print("RMSE:", rmse)
print("R2:", r2)

RMSE: 96281.52002316568
R2: 0.36895423179970455


In [28]:
linreg_model_log = LinearRegression()
linreg_model_log.fit(X_train_log, y_train_log)
prediction_log = linreg_model_log.predict(X_test_log)

# Model predicts log(price); exponentiate to convert back to ETB scale
# so RMSE/R² are directly comparable to the raw-price model
prediction_etb = np.exp(prediction_log)

rmse_log = np.sqrt(mean_squared_error(y_test, prediction_etb))
r2_log = r2_score(y_test, prediction_etb)

print("RMSE (log model, ETB scale):", rmse_log)
print("R2 (log model, ETB scale):", r2_log)

RMSE (log model, ETB scale): 104627.51047901923
R2 (log model, ETB scale): 0.2548104324008864


## 6. Ridge Regression
We repeat the same process using Ridge Regression (α=1.0), which adds regularization to reduce the risk of overfitting on our relatively small dataset.

In [11]:
ridge_model=Ridge(alpha=1.0)
ridge_model.fit(X_train,y_train)
prediction=ridge_model.predict(X_test)

rmse=np.sqrt(mean_squared_error(y_test,prediction))
r2=r2_score(y_test,prediction)

print("RMSE:", rmse)
print("R2:", r2)

RMSE: 96219.07921045518
R2: 0.3697724622083748


In [ ]:
ridge_model_log = Ridge(alpha=1.0)
ridge_model_log.fit(X_train_log, y_train_log)
prediction_log = ridge_model_log.predict(X_test_log)

# Model predicts log(price); exponentiate to convert back to ETB scale
# so RMSE/R² are directly comparable to the raw-price model
prediction_etb = np.exp(prediction_log)

rmse_log = np.sqrt(mean_squared_error(y_test, prediction_etb))
r2_log = r2_score(y_test, prediction_etb)

print("RMSE (log model, ETB scale):", rmse_log)
print("R2 (log model, ETB scale):", r2_log)

RMSE (log model, ETB scale): 104503.95461539131
R2 (log model, ETB scale): 0.2565693995290209


## 7. Model Coefficients & Interpretation
We extract the Ridge model's coefficients to see which features most strongly influence predicted rental price, and compare these against our EDA findings.

In [13]:
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": ridge_model.coef_  
}).sort_values("coefficient", ascending=False)

print(coef_df)

                           feature    coefficient
7              property_type_House   73639.963227
9              property_type_Villa   68071.843124
8          property_type_Penthouse   64239.681017
1                        furnished   46237.720363
0                         bedrooms   35104.568106
6             property_type_Duplex  -14355.364963
2            location_group_Kirkos  -23934.357789
4             location_group_Other  -32440.803864
3  location_group_Nifas Silk-Lafto  -49257.682457
5              location_group_Yeka -122410.983108


#### Interpretation:
The coefficients are interpreted relative to the dropped baseline category: an unfurnished Apartment in Bole.

Property type has the largest impact on price. Houses, Villas, and Penthouses command premiums of 64,000–74,000 ETB over a comparable apartment, reflecting the scarcity and size of standalone properties in Addis Ababa. Furnished units cost roughly 46,000 ETB more on average, and each additional bedroom adds about 35,000 ETB — both intuitive and consistent with EDA findings.

One anomaly worth noting: **Duplex** shows a negative coefficient (−14,355 ETB) relative to Apartment, despite typically being a larger property type. This likely reflects the small sample size for Duplex listings in this dataset (only a handful observed in EDA), making its coefficient unreliable rather than reflecting a genuine market pattern.

On the location side, Bole emerges as the most expensive subcity (the baseline), which aligns with EDA results. Kirkos and Nifas Silk-Lafto are moderately cheaper, while Yeka shows the largest negative coefficient (−122,411 ETB) — consistent with EDA where Yeka consistently ranked among the lowest-priced subcities.

Overall, the model's learned relationships are directionally sensible and match exploratory findings, giving us confidence that the features are capturing real pricing signals despite the modest R² of 0.37.

## 8. Model Results & Discussion

### Methodology
Two linear models were trained to predict rental prices in Addis Ababa: Linear Regression (baseline) and Ridge Regression (α=1.0). Features used were location group, property type, bedroom count, and furnished status. Categorical features were one-hot encoded with the first category dropped to avoid multicollinearity. A log-transformed target was also tested but ultimately performed worse on the ETB scale due to error amplification on high-price listings.

### Results
| Model | Target | RMSE (ETB) | R² |
|---|---|---|---|
| Linear Regression | Raw price | 96,282 | 0.369 |
| Linear Regression | Log price | 104,628 | 0.255 |
| Ridge (α=1.0) | Raw price | 96,219 | 0.370 |
| Ridge (α=1.0) | Log price | 104,504 | 0.257 |

Ridge Regression on raw price performed best, though marginally. The model explains roughly 37% of price variance — expected for a baseline model with only 4 features on noisy scraped data. Coefficient analysis confirmed that property type (House, Villa, Penthouse) and furnished status drive the largest price increases, while Yeka and Nifas Silk-Lafto subcities are significantly cheaper than the Bole baseline — consistent with EDA findings.

### Limitations
- Small dataset (~795 listings) is the primary constraint on model performance
- `size_m2` had to be dropped due to excessive missing values, despite being a strong price predictor
- Data was scraped from a single platform (Jiji.com.et), which may not represent the full Addis Ababa rental market
- Several subcities had too few listings to model individually and were grouped into "Other", losing location granularity
- The Duplex coefficient is negative and counterintuitive, likely an artifact of having only a few Duplex listings in the dataset rather than a real pricing pattern

### Future Work
- Train tree-based models (Random Forest, XGBoost) to capture non-linear relationships
- Expand the dataset with more scraping across additional platforms and subcities
- Engineer new features such as distance to city center or proximity to major roads
- Build an interactive Power BI dashboard to visualize predictions and EDA findings